<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Restart-From-Hackation_v1.0/mnps_post_getting_started%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNPS Job Equity Post Mini-Hackathon
> A notebook to help you get started  
> DSI DSSG + MNPS Hackathon  
> September 9, 2025  
> Drafted by Wayne Birch - [contact her](wayne.birch@mnps.org) for questions, code update needs, or other questions about the notebook!

This notebook is a restart point based on the work done in the mini Hackathon with Metro Nashville Public Schools (MNPS) and the VU Data Science Institute (VU DSI).

 **Competition Details from the Hackathon with some updates follow:**

You aren't constrained to what is in this notebook, and please feel free to use your creativity to deliver the best solution
# **1** | Competition Parameters
* **Outcome and evaluation:** Participants will be evaluated on the performance of their provided solution on the holdout set. Importantly, judges must be able to easily run the submitted code on the new dataset.
* **Objective:** The overall objective is to create a system which best automatically, reproducibly, and reliably categorizes jobs according to the parameters set forth by MNPS. A few suggestions are provided on parameters that you can vary if you're thinking about achievable changes in 2.5 hours



## **2** | Environment Setup
Again, you're completely free to just download this notebook, create a local virtual environment and get to coding in your favorite IDE. We provide this code just as a rapid method to get started, and focus our efforts on implementation through Google Colab.

### **2a** | API Key Setup
#### **2a.1** | Access
The DSI has provided you an API key which can access **some** of the OpenAI models. These include:
* All versions of gpt-4o
* All versions of gpt-4.1
* All versions of o3-mini

Vector store upload, web search, code interpreter, and other functionality outside of the Chat Completions and Messages API is **not** supported. If you really want to use these things, you will have to make a good and cost-supported argument. If you don't feel like arguing, you can also utilize your own OpenAI API key.

#### **2a.2** | API Keys in Google Colab
To use your API key, click on the key icon (looks sort of like 🔑) in the left sidebar.  Under **Name**, add `OPENAI_API_KEY`. Under **Value**, paste your API key. Your API key is a jumble of numbers and letters, maybe even other symbols. Click the slider checkbox to enable **Notebook access** (so your notebook will grab these values without asking you).  

### **2b** | Runtime setup
We're going to install some packages in your environment so that you have access to the code functionality. If you need more packages, install more packages. Install **only** packages you trust.

In [ ]:
!pip install openai

In [ ]:
# ===== Environment Setup (single source of truth) =====
import os
from typing import List
import pandas as pd
from pydantic import BaseModel, Field
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    fixes = {
        "gpt4o": "gpt-4o",
        "gpt-4o": "gpt-4o",
        "gpt4.1": "gpt-4.1",
        "gpt-41": "gpt-4.1",
        "o3mini": "o3-mini",
        "o3-mini": "o3-mini",
    }
    return fixes.get(s, s)

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-4o-2024-11-20"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


In [ ]:
# ==== UNIQUE RUN FOLDER + copy inputs + Responses API with attachments + save outputs ====
import os, json, shutil, datetime as dt
from pathlib import Path
import pandas as pd
from pydantic import BaseModel, Field
from typing import List
from google.colab import drive
from openai import OpenAI

# ---------- 0) Mount Drive ----------
drive.mount('/content/drive')

# ---------- 1) Create a NEW unique run folder each time ----------
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
timestamp = dt.datetime.utcnow().strftime("%Y%m%d_%H%M%S")  # UTC for portability
RUN_DIR = RUN_ROOT / f"RUN_{timestamp}"
INPUTS_DIR = RUN_DIR / "inputs"
OUTPUTS_DIR = RUN_DIR / "outputs"
for p in (RUN_DIR, INPUTS_DIR, OUTPUTS_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("🗂️ Run folder:", RUN_DIR)

# ---------- 2) Locate and COPY all input files used into this run's /inputs ----------
#    Edit SOURCE_INPUT_DIR if your files live elsewhere.
#    We try a few common candidates; the first that contains each file wins.
CANDIDATE_DIRS = [
    Path("/content/drive/My Drive/Colab Notebooks/Run Results"),          # common
    Path("/content/drive/My Drive/Colab Notebooks/Input Files"),          # optional
    Path("/content/drive/My Drive"),                                      # broad
    Path.cwd(),                                                           # current
]

file_names = [
    "Competency Extended Descriptions.csv",
    "Ground Truth Masterfile.csv",
    "Korn_Ferry Lominger 38 Competencies.csv",
    "MNPS KSACs.csv",
    "MNPS Roles.csv",
    "New Sample_08.07.2025.csv",
]

def find_file(filename: str, search_roots) -> Path | None:
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate
    return None

missing = []
copied_paths = []
for name in file_names:
    src = find_file(name, CANDIDATE_DIRS)
    if not src:
        missing.append(name)
        continue
    dst = INPUTS_DIR / name
    shutil.copy2(src, dst)
    copied_paths.append(dst)

if missing:
    raise FileNotFoundError(
        "These required input files were not found in the candidate folders:\n"
        + "\n".join(f" - {m}" for m in missing)
        + "\n\nPlease place them in one of the CANDIDATE_DIRS or update the list above."
    )

print("✅ Copied input files into:", INPUTS_DIR)
for p in copied_paths:
    print(" •", p.name)

# ---------- 3) Build one real test row (smoke test) from the sample CSV ----------
sample_csv = INPUTS_DIR / "New Sample_08.07.2025.csv"
df = pd.read_csv(sample_csv)

required_cols = [
    "Job Description Name","Position Summary","Education","Work Experience",
    "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in {sample_csv.name}: {missing_cols}")

ROW_IDX = 0  # change to try a different row
r = df.iloc[ROW_IDX]
job_desc_text = f"""Job Description Name: {r['Job Description Name']}

Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""

# ---------- 4) Pydantic schema (Structured Outputs) ----------
class JobClassification(BaseModel):
    job_title_original: str = Field(...)
    new_job_title: str = Field(...)
    major_role_group: str = Field(...)
    minor_sub_group: str = Field(...)
    grouping_justification: str = Field(...)

class JobClassificationTable(BaseModel):
    job_classification_table: List[JobClassification] = Field(...)
    narrative_rationale: str = Field(...)

# ---------- 5) Upload files to OpenAI (purpose='assistants') ----------
client = OpenAI()  # uses API key from your Environment Setup cell
uploaded = []
for p in copied_paths:
    with open(p, "rb") as f:
        up = client.files.create(file=f, purpose="assistants")
    uploaded.append(up)
file_ids = [u.id for u in uploaded]
print("⬆️ Uploaded file_ids:", file_ids)

# ---------- 6) Responses API call with attachments + Structured Outputs ----------
instructions = """You will classify the provided job description by FUNCTION (not title).
Use the attached CSV references (MNPS Roles, MNPS KSACs, Ground Truth, competencies) to inform:
- Major role grouping (e.g., Specialist, Analyst, Manager)
- Minor sub-grouping (I, II, III, IV — do not exceed IV)
Focus on qualitative, holistic alignment with KSACs and functional scope.
Return STRICTLY the JSON that conforms to the provided schema (table + narrative)."""

content = [{"type": "input_text", "text": instructions + "\n\n" + job_desc_text}]
for fid in file_ids:
    content.append({"type": "input_file", "file_id": fid})

print("🤖 Using model:", MODEL_ID)

# Preferred path: parse() → returns output_parsed (Pydantic) + output_text (JSON string)
try:
    resp = client.responses.parse(
        model=MODEL_ID,
        input=[{"role": "user", "content": content}],
        temperature=0.2,
        max_output_tokens=1400,
        response_format=JobClassificationTable,
    )
    parsed = resp.output_parsed
    raw_text = resp.output_text
except Exception as e:
    # Fallback to create() with explicit JSON schema (older SDKs)
    print("Fallback to responses.create() due to:", e)
    schema = JobClassificationTable.model_json_schema()
    resp = client.responses.create(
        model=MODEL_ID,
        input=[{"role": "user", "content": content}],
        temperature=0.2,
        max_output_tokens=1400,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "JobClassificationTable", "schema": schema, "strict": True},
        },
    )
    raw_text = getattr(resp, "output_text", None)
    if raw_text is None and hasattr(resp, "output") and resp.output:
        # try to stitch text segments if SDK returns chunks
        try:
            raw_text = "".join(getattr(seg, "text", "") for seg in resp.output_text)
        except Exception:
            raw_text = ""
    parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None

# ---------- 7) Save FINAL OUTPUTS + MANIFEST into the run folder ----------
RAW_PATH = OUTPUTS_DIR / "Raw_Response.json"
RAW_PATH.write_text(raw_text or "", encoding="utf-8")

written = {"raw_json": str(RAW_PATH)}
if parsed is not None:
    rows = [rec.model_dump() for rec in parsed.job_classification_table]
    csv_path = OUTPUTS_DIR / "Job_Classifications.csv"
    txt_path = OUTPUTS_DIR / "Narrative.txt"
    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8")
    txt_path.write_text(parsed.narrative_rationale, encoding="utf-8")
    written["table_csv"] = str(csv_path)
    written["narrative_txt"] = str(txt_path)
else:
    print("⚠️ No parsed object returned; saved Raw_Response.json only.")

# Manifest with reproducibility info
manifest = {
    "run_folder": str(RUN_DIR),
    "created_utc": timestamp,
    "model_used": os.environ.get("OPENAI_MODEL", "via MODEL_ID"),
    "resolved_model_id": globals().get("MODEL_ID", None),
    "source_inputs_copied": [str(p) for p in copied_paths],
    "uploaded_file_ids": file_ids,
    "sample_row_index": ROW_IDX,
    "outputs_written": written,
}
( RUN_DIR / "RUN_METADATA.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# ---------- 8) List contents for quick verification ----------
print("\n📁 Contents:")
for p in sorted(RUN_DIR.rglob("*")):
    print(" -", p.relative_to(RUN_DIR))


In [ ]:
from openai import OpenAI
client = OpenAI()

visible = {m.id for m in client.models.list().data}
if MODEL_ID not in visible:
    print(f"⚠️ {MODEL_ID} is not visible to your key. "
          "Use an alias you do see (e.g., gpt-4o) or confirm access in your org.")
else:
    print(f"👍 {MODEL_ID} is available.")


## **3** | The Data

The current prompt is a two-step prompt that is successful through the ChatGPT interface. It requires two types of data:
* The data to be classified
* Supporting resources

We need to read all of this in. Let's grab it and use it. The first thing you'll do is just straight up download a zip file of all of this information.

You can download all of the reference files from the link provided, then upload in the sidebar. You'll then unzip the directory using the code below.

Click on the folder icon in the left sidebar (kinda looks like this 🗂️) and you'll see all the files there. We'll read them in.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
base_target_folder = '/content/drive/My Drive/Colab Notebooks/Data Inputs'
!unzip "{base_target_folder}/MNPS_Prompt_Resources.zip" -d /content/

In [ ]:
# ===== Output folder on Google Drive =====
from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
import json

# Mount Drive (you'll be prompted once)
drive.mount('/content/drive')

# Where to save results (matches your structure)
RUN_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# New timestamped run folder, e.g., 20250909_175157
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / RUN_STAMP
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Saving outputs to:", RUN_DIR)


In [ ]:
import pandas as pd
resources_dir_prefix = '/content/'
roles_lookup = pd.read_csv(resources_dir_prefix+"MNPS Roles.csv")
determinants = pd.read_csv(resources_dir_prefix+"Competency Extended Descriptions.csv", encoding='latin1')
ksac_table = pd.read_csv(resources_dir_prefix+"MNPS KSACs.csv")
korn_ferry = pd.read_csv(resources_dir_prefix+"Korn_Ferry Lominger 38 Competencies.csv", encoding='latin1')

ground_truth_masterfile = pd.read_csv(f"{base_target_folder}/Ground Truth Masterfile.csv", encoding='latin1')
new_sample = pd.read_csv(f"{base_target_folder}/New Sample_08.07.2025.csv", encoding='latin1')

## **4** | The Prompts

What we have here is a direct prompt to get the response that we're looking for. We'll make this happen directly using the OpenAI Chat Completions API. Note that you can use other APIs as you like.

In [ ]:
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Compare each job with reference sources using the same attributes. I have attached the reference sources for you.
- Group jobs based on similarities into:
  - Major role groupings (e.g., Specialist, Analyst, Manager)
  - Minor sub-groupings (e.g., I, II, III, IV) - not to exceed level IV
- Use the MNPS Roles and MNPS KSACs documents to help you determine major role groupings.
- Use the remaining documents to help you clarify subtle differences in role groupings and sub-groupings.
- Use a more qualitative, holistic assessment focused on functional alignment with KSACs rather than a quantitative scoring approach with defined complexity metrics

Output Format:

- Create a table with the following columns:
  - Original Job Title
  - New Job Title
  - Major Role Group
  - Minor Sub-Group
  - Justification for Grouping

- Provide an accompanying narrative explaining the rationale behind the groupings and any notable patterns or insights discovered during the analysis.

Job Title Convention:

- Follow the format: "[Function] [Role] [Level]" (e.g., "Collections Specialist II", "Accounts Payable Specialist III")

Additional Guidelines:

- Ensure all sources used are cited properly.
- Focus on the nature of the work performed rather than just the job titles.
- Consider the complexity of tasks, level of responsibility, and required competencies when determining groupings.
- Provide clear explanations for why each job was classified as it was, referencing specific job attributes and external benchmarks.

"""

Instead of asking for a table output, we will use **structured outputs**. Though this is a common approach for the outputs of LLMs/AI systems, you can learn more about this on [OpenAI's structured output documentation](https://platform.openai.com/docs/guides/structured-outputs?api-mode=responses). Note that you can find this information on almost all LLM/AI platform or package providers.

In [ ]:
from pydantic import BaseModel, Field

class JobClassification(BaseModel):
    """Represents the classification of a job based on its functions."""
    job_title_original: str = Field(..., description="The original job title as provided in the input data using the job title convention specified.")
    new_job_title: str = Field(..., description="The proposed new job title based on the classification using the job title convention specified.")
    major_role_group: str = Field(..., description="The major grouping of the job based on its functional role (e.g., Specialist, Analyst, Manager).")
    minor_sub_group: str = Field(..., description="The minor sub-grouping within the major role group (e.g., Specialist I, II, III, IV).")
    grouping_justification: str = Field(..., description="The justification for placing the job in the specific major and minor groups, referencing job attributes and relevant documents.")

In [ ]:
from typing import List

class JobClassificationTable(BaseModel):
  """The table classification and overall commentary on the groupings provided by the AI system."""
  job_classification_table: List[JobClassification] = Field(..., description="The table of job classifications.")
  narrative_rationale: str = Field(..., description="The narrative commentary on the groupings provided by the AI system.")

Create classifications using OpenAI. Of note here is:
* The **developer** prompt - this is the "system prompt" or "custom instructions" for the model. This determines the overall behavior of the model.
* The **user** prompt - this is what we send to the model like when we're chatting with ChatGPT.

In [ ]:
# ===== Create classifications (Responses API + attachments, saves to OUTPUTS_DIR) =====
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
from pathlib import Path
import pandas as pd
import json
import os

client = OpenAI()  # API key set in Environment Setup

# ---- Pydantic schema (same as earlier) ----
class JobClassification(BaseModel):
    job_title_original: str = Field(...)
    new_job_title: str = Field(...)
    major_role_group: str = Field(...)
    minor_sub_group: str = Field(...)
    grouping_justification: str = Field(...)

class JobClassificationTable(BaseModel):
    job_classification_table: List[JobClassification] = Field(...)
    narrative_rationale: str = Field(...)

# ---- Ensure we have file_ids (reuse from earlier cell, or upload from INPUTS_DIR) ----
try:
    file_ids  # type: ignore
except NameError:
    # Upload all CSVs inside INPUTS_DIR
    assert 'INPUTS_DIR' in globals(), "INPUTS_DIR not found. Run the unique-run cell first."
    uploaded = []
    for p in Path(INPUTS_DIR).glob("*.csv"):
        with open(p, "rb") as f:
            up = client.files.create(file=f, purpose="assistants")
        uploaded.append(up)
    file_ids = [u.id for u in uploaded]
    print("⬆️ Uploaded file_ids:", file_ids)

# ---- Build one job description payload ----
# If you already have a variable `job_desc_text`, we’ll use it; else read row 0 from the sample CSV.
if 'job_desc_text' not in globals():
    sample_csv = Path(INPUTS_DIR) / "New Sample_08.07.2025.csv"
    df = pd.read_csv(sample_csv)
    required_cols = [
        "Job Description Name","Position Summary","Education","Work Experience",
        "Essential Functions","Licenses and Certifications","Knowledge, Skills and Abilities"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {sample_csv.name}: {missing}")
    r = df.iloc[0]
    job_desc_text = f"""Job Description Name: {r['Job Description Name']}

Position Summary: {r['Position Summary']}
Education: {r['Education']}
Work Experience: {r['Work Experience']}
Licenses and Certifications: {r['Licenses and Certifications']}
Essential Functions: {r['Essential Functions']}
Knowledge, Skills and Abilities: {r['Knowledge, Skills and Abilities']}
"""

# ---- Instructions (short, schema-focused) ----
instructions = """Classify the job by FUNCTION (not title).
Use attached CSV references (MNPS Roles, MNPS KSACs, Ground Truth, competencies).
Return STRICT JSON that matches the provided schema: table + narrative.
"""

# ---- Build Responses API content with attachments ----
content = [{"type": "input_text", "text": instructions + "\n\n" + job_desc_text}]
for fid in file_ids:
    content.append({"type": "input_file", "file_id": fid})

# ---- Call Responses API with Structured Outputs (Pydantic) ----
assert 'MODEL_ID' in globals(), "MODEL_ID not set. Run Environment Setup first."
print("🤖 Using model:", MODEL_ID)

try:
    resp = client.responses.parse(
        model=MODEL_ID,
        input=[{"role": "user", "content": content}],
        temperature=0.2,
        max_output_tokens=1400,
        response_format=JobClassificationTable,
    )
    parsed = resp.output_parsed
    raw_text = resp.output_text
except Exception as e:
    print("Fallback to responses.create() due to:", e)
    schema = JobClassificationTable.model_json_schema()
    resp = client.responses.create(
        model=MODEL_ID,
        input=[{"role": "user", "content": content}],
        temperature=0.2,
        max_output_tokens=1400,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "JobClassificationTable", "schema": schema, "strict": True},
        },
    )
    # Try to extract raw text from the response object
    raw_text = getattr(resp, "output_text", None) or ""
    parsed = JobClassificationTable.model_validate_json(raw_text) if raw_text else None

# ---- Save outputs to OUTPUTS_DIR ----
assert 'OUTPUTS_DIR' in globals(), "OUTPUTS_DIR not found. Run the unique-run cell first."
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

raw_path = Path(OUTPUTS_DIR) / "Raw_Response.json"
raw_path.write_text(raw_text or "", encoding="utf-8")

if parsed is not None:
    rows = [row.model_dump() for row in parsed.job_classification_table]
    out_csv = Path(OUTPUTS_DIR) / "Job_Classifications.csv"
    pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8")

    out_txt = Path(OUTPUTS_DIR) / "Narrative.txt"
    out_txt.write_text(parsed.narrative_rationale, encoding="utf-8")

    print("✅ Saved:", out_csv)
    print("✅ Saved:", out_txt)
else:
    print("⚠️ No parsed Structured Output returned; saved Raw_Response.json only at:", raw_path)

print("✅ Saved:", raw_path)

# ---- Console visibility (optional) ----
print("\n=== RAW JSON STRING FROM MODEL ===")
print(raw_text or "(empty)")
if parsed is not None:
    print("\n=== PARSED (Pydantic) ===")
    print(parsed.model_dump_json(indent=2))

# ---- List run outputs ----
print("\nContents of OUTPUTS_DIR:")
for p in sorted(Path(OUTPUTS_DIR).glob("*")):
    print(" -", p.name)


In [ ]:
#look at response
response.choices[0].message.parsed

We can make this into a table using pandas!

In [ ]:
response_dict = dict(*response.choices[0].message.parsed.job_classification_table)
response_dict

In [ ]:
# see outputs
pd.DataFrame(response_dict, index=[0])

In [ ]:
import os
import datetime
import shutil
import ipykernel

# Get the notebook name
try:
    # This method works in Colab
    notebook_path = ipykernel.get_connection_file()
    notebook_name = os.path.basename(notebook_path).split('.')[0]
except:
    # Fallback for other environments
    notebook_name = 'Colab_Notebook_Run'

# Define the destination directory in Google Drive
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
destination_dir = f'/content/drive/My Drive/Colab Notebooks/Run Results/{timestamp}'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# List of files to copy (modify this list as needed)
files_to_copy = [
    '/content/Korn_Ferry Lominger 38 Competencies.csv',
    '/content/Competency Extended Descriptions.csv',
    '/content/MNPS KSACs.csv',
    '/content/MNPS Roles.csv',
    f'{base_target_folder}/Ground Truth Masterfile.csv', # Copying from the original location
    f'{base_target_folder}/New Sample_08.07.2025.csv', # Copying from the original location
    # Add any other files you want to copy from the run, e.g., output files
    # '/content/your_output_file.csv'
]

# Copy the files
for file_path in files_to_copy:
    try:
        shutil.copy(file_path, destination_dir)
        print(f"Copied: {file_path} to {destination_dir}")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"Error copying {file_path}: {e}")

print("File copying complete.")